# 11 · Build the organizer

**Step 1.** `10_simulate_data` left files in three unrelated trees and a dict
describing only some of them. This notebook builds the one object that knows
where all of it is.

```
Organizer("~/Repos/OrgDemo/Lab/lab.json")
 └── sample_id              "S03"
      ├── stages            synthesis — the conditions, from the dict
      ├── measurements      uvvis (ingested) · tem, waxs1d (linked by hand)
      └── derived           filled in by 12
```

The organizer is **one JSON file you name**. Not a directory layout, not a
hidden folder — a document you can copy, diff, version and email. The data
stays where it is; the file only records where that is.

Two ways in, and they are for different situations:

| | |
|---|---|
| **ingest** | metadata somebody already wrote — a dict or a `*_dict.py`. Bulk, structured, re-readable. |
| **link** | data nobody wrote down. One call per measurement, by hand, because that is the only thing that is true. |

There is deliberately **no auto-attach here**. `attach_folders()` exists and
works, but it needs `<Modality>Data/<SampleID>/` under one root — a layout
almost nobody has. Building a demo on it teaches a habit that breaks on
contact with a real campaign.

In [1]:
from pathlib import Path
import json

from NanoOrganizer import Organizer
from NanoOrganizer.demo import demo_root

LAB = demo_root("Lab")
RAW = LAB / "RawData"

if not (LAB / "Meta" / "synthesis_dict.json").exists():
    raise SystemExit("Run 10_simulate_data first.")

org = Organizer(LAB / "lab.json", name="lab demo")
org

<Organizer 'lab demo': 0 samples, all — /home/yuzhang/Repos/OrgDemo/Lab/lab.json>

## Ingest the metadata dict

`ingest` takes the **dict itself**, not a path to it. The keyword names the
stage, which matters because a dict's variable name is invisible once it has
been passed in — `org.ingest(synthesis=d)` says what `org.ingest(d)` cannot.

Any block in a record that names files becomes a measurement; everything else
is kept verbatim as a stage parameter and flattens into a dotted column.

In [2]:
Synthesis_dict = json.loads((LAB / "Meta" / "synthesis_dict.json").read_text())

org.ingest(synthesis=Synthesis_dict)
print(org.summary())

org.table()[["synthesis.conditions.temperature_C", "synthesis.status",
             "modalities", "n_measurements"]]

lab demo: 6 samples, 6 measurements (6 readable here, 0 not mounted).
Modalities: uvvis
Root: /home/yuzhang/Repos/OrgDemo/Lab


,synthesis.conditions.temperature_C,synthesis.status,modalities,n_measurements
sample_id,,,,
S01,60.0,done,uvvis,1
S02,70.0,done,uvvis,1
S03,80.0,done,uvvis,1
S04,90.0,done,uvvis,1
S05,100.0,done,uvvis,1
S06,110.0,done,uvvis,1


Six samples, one measurement each — the spectra, because that is all the dict
mentioned. The micrographs and the diffraction are still invisible.

## Editing the dict and re-ingesting

The dict is live. Append a sample, fix a number, pop a mistake — then ingest
again. `replace=True` makes the dict the **whole truth** for its stage:
parameters are replaced rather than merged, and a sample that has left the
dict loses that stage. Without it a merge can only add, so a popped key would
sit there forever looking like data.

In [3]:
# A seventh synthesis that failed: it has conditions and no data at all.
# Recording it matters — leaving it out biases every later comparison.
Synthesis_dict["S07"] = {
    "sample_id": "S07",
    "synthesis_batch": {"run_id": "run_007", "status": "error",
                        "error": "precursor precipitated"},
    "conditions": {"temperature_C": 120.0, "hold_time_min": 21.0,
                   "stir_rate_rpm": 400, "solvent": "water"},
}

# A transcription error, found later.
Synthesis_dict["S02"]["conditions"]["temperature_C"] = 72.5

org.ingest(synthesis=Synthesis_dict, replace=True)

org.table()[["synthesis.conditions.temperature_C", "synthesis.status",
             "n_measurements"]]

,synthesis.conditions.temperature_C,synthesis.status,n_measurements
sample_id,,,
S01,60.0,done,1
S02,72.5,done,1
S03,80.0,done,1
S04,90.0,done,1
S05,100.0,done,1
S06,110.0,done,1
S07,120.0,error,0


In [4]:
# And a sample that turns out to belong to someone else's campaign.
Synthesis_dict.pop("S06")
org.ingest(synthesis=Synthesis_dict, replace=True)
print(org.project.sample_ids())

# Put it back — 12 wants all six.
Synthesis_dict["S06"] = json.loads(
    (LAB / "Meta" / "synthesis_dict.json").read_text())["S06"]
org.ingest(synthesis=Synthesis_dict, replace=True)
print(org.project.sample_ids())

['S01', 'S02', 'S03', 'S04', 'S05', 'S07']
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07']


## Link what nobody wrote down

Now the two instruments that never made it into the dict. `link` takes a
sample id, a technique, and where the data is — and reads the third argument
three ways:

| | |
|---|---|
| a **folder** | listed now, filtered by the technique's extensions — a snapshot you can audit |
| a **glob** | kept live, re-expanded at read time, so frames written later appear |
| a **path or list** | stored verbatim |

The micrographs are a folder per sample; the diffraction is one file each.

In [5]:
for sample in org.project.sample_ids():
    scope = RAW / "microscope_share" / sample
    waxs = RAW / "xrd_rig" / f"{sample}_waxs.dat"

    # S07 failed, so neither exists for it. A link to nothing is a mistake,
    # not a feature: skip it and let the gap be visible in the catalog.
    if scope.is_dir():
        org.link(sample, "tem", str(scope),
                 stage="characterization", instrument="LabScope",
                 nm_per_pixel=0.5)
    if waxs.exists():
        org.link(sample, "waxs1d", str(waxs),
                 stage="characterization", instrument="LabXRD")

print(org.summary())

lab demo: 7 samples, 18 measurements (18 readable here, 0 not mounted).
Modalities: uvvis, tem, waxs1d
Root: /home/yuzhang/Repos/OrgDemo/Lab


In [6]:
# session.txt sits beside the micrographs and is not a micrograph: the
# technique's declared extensions left it out without being asked.
tem = org.measurement("S01", modality="tem")
print(len(tem.paths), "files:", [Path(p).name for p in tem.paths])

3 files: ['img_01.tif', 'img_02.tif', 'img_03.tif']


## What is in it, and what is not

The gaps are the point. A measurement matrix is always sparse, and seeing
which comparisons exist beats discovering halfway through that the argument
rests on a sample that was never measured.

In [7]:
org.catalog()

,uvvis,tem,waxs1d
sample_id,,,
S01,True,True,True
S02,True,True,True
S03,True,True,True
S04,True,True,True
S05,True,True,True
S06,True,True,True
S07,False,False,False


## Paths are recorded as given

Nothing is rewritten relative to the organizer — a rewritten store only works
on the machine that wrote it. Portability is an alias instead: each link
registers the mount its data sits on, so another machine edits one line per
mount rather than one per record.

```python
org.project.add_alias("/mnt/scope", ["/Volumes/scope"])    # elsewhere
```

In [8]:
m = org.measurement("S01", modality="waxs1d")
print("recorded:", m.paths[0])
print("resolves:", m.resolve(org.resolver)[0])

for alias in org.project.config.path_aliases:
    print("alias:", alias.prefix, "->", ", ".join(alias.candidates))

report = org.available()
print(f"\n{report['n_available']}/{report['n_measurements']} readable here")

recorded: /home/yuzhang/Repos/OrgDemo/Lab/RawData/xrd_rig/S01_waxs.dat
resolves: /home/yuzhang/Repos/OrgDemo/Lab/RawData/xrd_rig/S01_waxs.dat
alias: /home/yuzhang -> /home/yuzhang

18/18 readable here


## Anything else worth filtering on

`set_params` records conditions that were never in the dict — an operator's
note, a correction, a grouping you only thought of afterwards.

In [9]:
for sample in ("S01", "S02", "S03"):
    org.set_params(sample, stage="synthesis", batch_group="morning")
for sample in ("S04", "S05", "S06"):
    org.set_params(sample, stage="synthesis", batch_group="afternoon")

org.table()[["synthesis.conditions.temperature_C", "synthesis.batch_group"]]


,synthesis.conditions.temperature_C,synthesis.batch_group
sample_id,,
S01,60.0,morning
S02,72.5,morning
S03,80.0,morning
S04,90.0,afternoon
S05,100.0,afternoon
S06,110.0,afternoon
S07,120.0,NaN


## Save

One file. Links, parameters, aliases and (later) derived values, all in it.
The data is untouched and stays where it was.

In [10]:
print(org.save())
print(f"{org.path.stat().st_size / 1024:.1f} kB")

/home/yuzhang/Repos/OrgDemo/Lab/lab.json
23.0 kB


In [11]:
# A links table is the other export — one row per measurement, re-importable
# with org.link_table(). A spreadsheet is a better editor than a form when
# forty rows need the same fix.
org.links_table().head(4)

,sample_id,modality,source,stage,role,instrument,n_frames,spectrum_glob,nm_per_pixel
0,S01,uvvis,/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectr...,synthesis,growth,LabSpec 2000,14.0,/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectr...,NaN
1,S01,tem,/home/yuzhang/Repos/OrgDemo/Lab/RawData/micros...,characterization,,LabScope,NaN,NaN,0.5
2,S01,waxs1d,/home/yuzhang/Repos/OrgDemo/Lab/RawData/xrd_ri...,characterization,,LabXRD,NaN,NaN,NaN
3,S02,uvvis,/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectr...,synthesis,growth,LabSpec 2000,14.0,/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectr...,NaN


---

**Next: `12_use_organizer`** — reopen this file and do the work: filter,
visualise, fit, and link the results back so they can be read again without
refitting.